**Como está o catálogo de ofertas dos fornecedores, quais modelos têm mais estoque disponível, preço médio por marca, e quantos fornecedores oferecem cada modelo?**

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.silver;

CREATE OR REPLACE TABLE workspace.silver.offer AS
SELECT id, fk_supplier, fk_model, unit_price, availability, expiration_date
FROM solaria.public.offer
WHERE unit_price > 0
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY expiration_date DESC NULLS LAST) = 1;

CREATE OR REPLACE TABLE workspace.silver.inventory AS
SELECT id, fk_supplier, fk_model, quantity
FROM solaria.public.inventory
QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY quantity DESC) = 1;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

CREATE OR REPLACE TABLE workspace.gold.dim_modelo AS
SELECT
    id AS model_id,
    brand,
    model AS nome_modelo,
    power_wp,
    efficiency,
    status
FROM solaria.public.model;

CREATE OR REPLACE TABLE workspace.gold.dim_fornecedor_catalogo AS
SELECT
    s.id AS supplier_id,
    c.trade_name AS empresa
FROM solaria.public.supplier s
JOIN solaria.public.company c ON c.id = s.fk_company;

CREATE OR REPLACE TABLE workspace.gold.fato_ofertas AS
SELECT id AS offer_id, fk_supplier, fk_model, unit_price, availability, expiration_date
FROM workspace.silver.offer;

CREATE OR REPLACE TABLE workspace.gold.fato_estoque AS
SELECT id AS inventory_id, fk_supplier, fk_model, quantity
FROM workspace.silver.inventory;

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.kpi_estoque_por_modelo AS
SELECT
    dm.brand,
    dm.nome_modelo,
    SUM(fe.quantity) AS estoque_total,
    RANK() OVER (ORDER BY SUM(fe.quantity) DESC) AS ranking_estoque
FROM workspace.gold.fato_estoque fe
JOIN workspace.gold.dim_modelo dm ON dm.model_id = fe.fk_model
GROUP BY dm.brand, dm.nome_modelo;

CREATE OR REPLACE TABLE workspace.gold.kpi_preco_medio_por_marca AS
SELECT
    dm.brand,
    AVG(fo.unit_price) AS preco_medio,
    STDDEV(fo.unit_price) AS desvio_preco
FROM workspace.gold.fato_ofertas fo
JOIN workspace.gold.dim_modelo dm ON dm.model_id = fo.fk_model
GROUP BY dm.brand;

CREATE OR REPLACE TABLE workspace.gold.kpi_fornecedores_por_modelo AS
SELECT
    dm.brand,
    dm.nome_modelo,
    COUNT(DISTINCT fo.fk_supplier) AS total_fornecedores,
    RANK() OVER (ORDER BY COUNT(DISTINCT fo.fk_supplier) DESC) AS ranking
FROM workspace.gold.fato_ofertas fo
JOIN workspace.gold.dim_modelo dm ON dm.model_id = fo.fk_model
GROUP BY dm.brand, dm.nome_modelo;